In [1]:
# ==============================
# Federated Learning on EMNIST Digits (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedAvg
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1  # very strict non-IID

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: EMNIST Digits
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.EMNIST(
    root="./data", split="digits", train=True, download=True, transform=transform
)
test_dataset = datasets.EMNIST(
    root="./data", split="digits", train=False, download=True, transform=transform
)

# -------------------------
# Validation Split: 10 fixed clients
# -------------------------
VAL_CLIENTS = list(range(10))  # first 10 clients for validation
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))  # remaining 90 clients for training

# -------------------------
# Dirichlet-based Non-IID Partition
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):
    """
    Returns dict: client_id -> list of indices
    alpha: small -> very strict non-IID
    """
    num_classes = 10
    labels = np.array(dataset.targets)
    idxs = np.arange(len(labels))

    client_dict = {i: [] for i in clients}

    # split indices by class
    class_indices = [idxs[labels == i] for i in range(num_classes)]

    for c_idx in class_indices:
        np.random.shuffle(c_idx)
        proportions = np.random.dirichlet(alpha*np.ones(len(clients)))
        proportions = (np.cumsum(proportions) * len(c_idx)).astype(int)
        proportions = np.concatenate(([0], proportions))
        for i, client in enumerate(clients):
            client_dict[client] += list(c_idx[proportions[i]:proportions[i+1]])

    # Shuffle each client's indices
    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

# Apply Dirichlet partition
client_data = dirichlet_partition(train_dataset, list(range(NUM_CLIENTS)), alpha=ALPHA)

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_val = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_val += F.cross_entropy(out, y).item() * y.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100*correct/total, loss_val/total

# -------------------------
# Federated Training
# -------------------------
global_model = EMNIST_CNN().to(DEVICE)
scaler = torch.cuda.amp.GradScaler()

# Metrics lists
train_acc, val_acc, test_acc = [], [], []
loss_curve, train_loss_curve, val_loss_curve, test_loss_curve = [], [], [], []
round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):
    start_round = time.time()
    local_weights = []
    local_sizes = []
    local_losses = []

    # select clients randomly for this round
    selected_clients = random.sample(TRAIN_CLIENTS, int(NUM_CLIENTS*CLIENT_FRAC))

    for client in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = optim.SGD(local_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=LOCAL_EPOCHS)

        loader = DataLoader(Subset(train_dataset, client_data[client]), batch_size=BATCH_SIZE, shuffle=True)

        for _ in range(LOCAL_EPOCHS):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    out = local_model(x)
                    loss = F.cross_entropy(out, y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

        local_losses.append(loss.item())
        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(client_data[client]))

    # Weighted aggregation
    total = sum(local_sizes)
    new_weights = OrderedDict()
    for k in global_model.state_dict().keys():
        new_weights[k] = sum(
            local_weights[i][k]*(local_sizes[i]/total) for i in range(len(local_weights))
        )
    global_model.load_state_dict(new_weights)

    # Compute metrics
    # Train
    train_loader_all = DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in selected_clients]),
        batch_size=BATCH_SIZE, shuffle=False
    )
    tr_acc, tr_loss = evaluate(global_model, train_loader_all)

    # Validation
    val_loader_all = DataLoader(
        torch.utils.data.ConcatDataset([Subset(train_dataset, client_data[c]) for c in VAL_CLIENTS]),
        batch_size=BATCH_SIZE, shuffle=False
    )
    va_acc, va_loss = evaluate(global_model, val_loader_all)

    # Test
    te_acc, te_loss = evaluate(global_model, DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False))

    # Save metrics
    loss_curve.append(np.mean(local_losses))
    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)
    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    round_t = time.time() - start_round
    round_time.append(round_t)
    total_time.append(time.time() - start_total)

    print(f"Round {rnd+1:03d} | "
          f"LocalLoss {loss_curve[-1]:.4f} | "
          f"TrainLoss {train_loss_curve[-1]:.4f} | "
          f"ValLoss {val_loss_curve[-1]:.4f} | "
          f"TestLoss {test_loss_curve[-1]:.4f} | "
          f"Train {train_acc[-1]:.4f} | "
          f"Val {val_acc[-1]:.4f} | Test {test_acc[-1]:.4f} | "
          f"Round Time {round_time[-1]:.2f}s | Total Time {total_time[-1]:.2f}s")

Device: cuda


/tmp/ipykernel_1548/2795009074.py:137: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1548/2795009074.py:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Round 001 | LocalLoss 0.4710 | TrainLoss 1.7282 | ValLoss 2.0170 | TestLoss 1.9482 | Train 37.7443 | Val 23.5182 | Test 26.7800 | Round Time 206.25s | Total Time 206.25s
Round 002 | LocalLoss 0.1605 | TrainLoss 1.1381 | ValLoss 1.6291 | TestLoss 1.6544 | Train 62.9353 | Val 46.4635 | Test 48.5575 | Round Time 142.28s | Total Time 348.53s
Round 003 | LocalLoss 0.1428 | TrainLoss 1.2989 | ValLoss 1.3099 | TestLoss 1.2407 | Train 57.4052 | Val 55.8722 | Test 57.2075 | Round Time 124.09s | Total Time 472.63s
Round 004 | LocalLoss 0.1294 | TrainLoss 0.7848 | ValLoss 0.6853 | TestLoss 0.8427 | Train 80.9657 | Val 83.9699 | Test 79.0075 | Round Time 103.56s | Total Time 576.18s
Round 005 | LocalLoss 0.0749 | TrainLoss 0.4501 | ValLoss 0.8126 | TestLoss 0.6583 | Train 85.5661 | Val 71.8950 | Test 77.5700 | Round Time 92.39s | Total Time 668.57s
Round 006 | LocalLoss 0.0665 | TrainLoss 0.5328 | ValLoss 0.4921 | TestLoss 0.5782 | Train 84.0522 | Val 85.3103 | Test 82.5000 | Round Time 100.58s | 

/tmp/ipykernel_1548/2795009074.py:175: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Round 052 | LocalLoss 0.0146 | TrainLoss 0.0485 | ValLoss 0.0587 | TestLoss 0.0602 | Train 98.5184 | Val 98.0867 | Test 98.0425 | Round Time 68.74s | Total Time 4614.50s
Round 053 | LocalLoss 0.0073 | TrainLoss 0.0503 | ValLoss 0.0525 | TestLoss 0.0581 | Train 98.5009 | Val 98.3144 | Test 98.2325 | Round Time 94.00s | Total Time 4708.50s
Round 054 | LocalLoss 0.0076 | TrainLoss 0.0511 | ValLoss 0.0600 | TestLoss 0.0641 | Train 98.4857 | Val 98.1528 | Test 98.0400 | Round Time 89.29s | Total Time 4797.79s
Round 055 | LocalLoss 0.0091 | TrainLoss 0.0466 | ValLoss 0.0522 | TestLoss 0.0537 | Train 98.6336 | Val 98.3217 | Test 98.3625 | Round Time 72.17s | Total Time 4869.96s
Round 056 | LocalLoss 0.0086 | TrainLoss 0.0444 | ValLoss 0.0517 | TestLoss 0.0519 | Train 98.6689 | Val 98.3621 | Test 98.3975 | Round Time 84.72s | Total Time 4954.68s
Round 057 | LocalLoss 0.0153 | TrainLoss 0.0514 | ValLoss 0.0665 | TestLoss 0.0694 | Train 98.3842 | Val 97.8810 | Test 97.8425 | Round Time 84.57s | 

In [2]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("EMNIST_Digits_fedavg_federated_metrics_0.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("EMNIST_Digits_fedavg_federated_metrics_0.csv")
except:
    print("CSV saved locally as EMNIST_Digits_fedavg_federated_metrics_0.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>